# European privacy fines: reproduce the published snapshot

[Study and source notes](https://www.legiscope.com/blog/amendes-rgpd-europe-comparaison-pays.html). Original Legiscope contributions: CC BY 4.0; code: MIT.

The collection has unequal periods, mixed legal regimes and unknown amounts. Its per-resident figures do not rank national enforcement severity. This notebook runs with the Python standard library and contains outputs from actual execution.

In [1]:
import csv, io, json
from pathlib import Path
from decimal import Decimal
candidates = [p / 'data/dataset.json' for p in (Path.cwd(), *Path.cwd().parents)]
candidates += [Path.cwd() / 'dataset.json']
if Path('/kaggle/input').exists():
    candidates += sorted(Path('/kaggle/input').rglob('dataset.json'))
dataset_path = next((p for p in candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError('Extract the release or attach the dataset first.')
data = json.loads(dataset_path.read_text(encoding='utf-8'))
cases = data['cases']
print('Dataset:', data['meta']['version'])
print('Cases:', len(cases), '| Countries:', len(data['recent']))

Dataset: 2026-10-30-source-audit
Cases: 1042 | Countries: 10


## Country totals and monetary coverage

Recompute from the case records with Decimal arithmetic. Missing EUR amounts are excluded from the sum and counted explicitly. The denominator is the recorded population on 1 January 2026.

In [2]:
summary_rows = []
for country in data['recent']:
    rows = [r for r in cases if r['country_code'] == country['country_code']]
    known = [r for r in rows if r.get('amount_eur') is not None]
    total = sum((Decimal(str(r['amount_eur'])) for r in known), Decimal('0')) if known else None
    ratio = total / Decimal(str(country['population'])) if total is not None else None
    if total is not None and total != Decimal(str(country['amount_eur'])):
        raise ValueError('Published country total differs')
    summary_rows.append(dict(country=country['country_code'], decisions=len(rows), known=len(known),
                             unknown=len(rows)-len(known), documented_eur=str(total),
                             population=country['population'],
                             eur_per_resident=str(ratio.quantize(Decimal('0.000001'))) if ratio is not None else None))
stream = io.StringIO()
writer = csv.DictWriter(stream, fieldnames=list(summary_rows[0]))
writer.writeheader()
writer.writerows(summary_rows)
print(stream.getvalue().strip())

country,decisions,known,unknown,documented_eur,population,eur_per_resident
FR,200,200,0,661782000.0,69112309,9.575458
DE,102,43,59,50826200.0,83467117,0.608937
ES,134,134,0,17597950.0,49590099,0.354868
LT,40,37,3,3263038.0,2887592,1.130020
PL,141,135,6,19160846.02,36332765,0.527371
LV,38,37,1,1833059.93,1845096,0.993477
IT,200,192,8,37214293.0,58942828,0.631363
EE,2,2,0,3005000.0,1360745,2.208349
LU,55,55,0,573736.0,690959,0.830347
GR,130,130,0,37399749.0,10366586,3.607721


## Findings in the French selection

Count distinct cases per theme within France. One case may have several themes. These frequencies cannot be compared as national offence rates. Never sum money after joining the multi-theme bridge.

In [3]:
from collections import Counter
france = [r for r in cases if r['country_code'] == 'FR']
counts = Counter(theme for row in france for theme in sorted(set(row['themes'])))
labels = {t['id']: t['label'] for t in data['themes']}
for theme, count in counts.most_common(10):
    print(f'{count:3} / {len(france)}  {labels[theme]}')

 66 / 200  Mal informer les personnes
 54 / 200  Ne pas coopérer avec l’autorité de contrôle
 53 / 200  Protéger insuffisamment les données
 50 / 200  Ne pas respecter les demandes des personnes
 41 / 200  Ne pas respecter les conditions légales d’utilisation
 41 / 200  Collecter ou surveiller de façon excessive
 37 / 200  Ne pas respecter les règles sur les cookies
 34 / 200  Conserver les données trop longtemps
 19 / 200  Ne pas respecter les obligations de sous-traitance
 17 / 200  Mal recueillir ou gérer le consentement


## Trace one observation to its source

Original amounts, retained EUR values, appeal notes and source depth are distinct fields. Source links allow further review; their presence does not certify finality or complete reasoning review.

In [4]:
example = cases[0]
for key in ['id', 'country_code', 'organisation', 'date', 'date_kind', 'amount_original',
            'currency', 'amount_eur', 'source_level', 'appeal_note', 'source_url', 'decision_url']:
    print(f'{key}: {example.get(key)}')

id: cnil-2026-supplement-003
country_code: FR
organisation: EXTIA
date: 2026-07-21
date_kind: decision
amount_original: 300000
currency: EUR
amount_eur: 300000.0
source_level: operative_provisions_checked
appeal_note: Recours et paiement non vérifiés. Montant initial hors astreintes.
source_url: https://www.cnil.fr/fr/sanction-non-respect-droits-personnes-extia
decision_url: https://www.legifrance.gouv.fr/cnil/id/CNILTEXT000054810440


## Reuse

Cite [Legiscope and the study](https://www.legiscope.com/blog/amendes-rgpd-europe-comparaison-pays.html), identify this frozen release and retain the coverage and source notes. See `docs/methodology.md` and `LICENSING.md` in the repository.